# 分析自己的磁结构结果

推荐先看下面的个人结果用法；它不需要生成演示数据，也不重新求解。

```python
frames = pp.frames_from_state(Path('你的输出名.state'))
results = pp.analyze(frames, assume_temporal_sampling=True, ab_reference='antiparallel')
results.show()
```

state必须绑定完整且未改动的CSV/OVF库存；自动入口只适用于可唯一绑定的最后阶段。已有结果读取有分阶段进度。两项假设须符合自己的数据，不能机械照搬。

`analyze`返回可show/plot/to_csv的Analysis；旧`analyze_ovf`返回列表。show只展示常用指标，完整数据用to_csv导出。

以下各单元是**可选的离线合成演示**，不读取个人磁态、不运行LLG。目录必须是新名字，重复运行不会覆盖。


In [ ]:
import json
from pathlib import Path
import fimrun.postprocess as pp


## 1. 明确的合成输入

`generate_demo`确实写OVF、inputs.json和analysis.csv，返回旧接口的rows。新入口接收inputs.json中的A/B路径，不接收返回rows。时间、边界、segment和反平行参考在解析例中已知；真实数据不能照搬这些假设。


In [ ]:
example_dir = Path('new-skyrmion-example')
pp.generate_demo(example_dir)
frames = json.loads((example_dir / 'inputs.json').read_text())


## 一次分析，重复看图

分析只运行一次。show默认精简概要、常用图和折叠详情；每个指标的有效性独立，影响当前指标的警告不会隐藏。table/plot/show只读已有结果，不重新求解。

`assume_temporal_sampling=True`声明相邻帧位移小于半盒，用于周期轨迹和速度；`ab_reference="antiparallel"`仅为A/B相位比较选择参考，不修改磁矩。


In [ ]:
result = pp.analyze(frames, assume_temporal_sampling=True,
                    ab_reference='antiparallel')
result.show()


## 3. 面积、半径与形状

$$R=\sqrt{A/\pi},\qquad a=2\sqrt{\lambda_{\max}},\quad b=2\sqrt{\lambda_{\min}}.$$

A为轮廓面积，a/b是面积协方差定义的半轴，不是全轴长。表内SI；图中长度nm。`valid_geometry`与`valid_axis`分开，近圆方向NaN是预期。CSV字段为A/B_area_m2、radius_m、major_m/minor_m。


In [ ]:
result.table(metrics=['radius', 'shape', 'axis'], sublattice='both')


In [ ]:
result.plot('radius', sublattice='both')


## 4. 拓扑、相位与A/B

$$Q=\frac{1}{4\pi}\sum_T\Omega_T,\qquad Z=\frac1N\sum_k e^{\mathrm{i}(\psi_k-n\phi_k)},\qquad\chi=\arg Z.$$

有符号三角形决定Q；沿轮廓等弧长采样，n由CCW绕数测得，不能用Q符号代替。`chi_raw`只诊断，`chi_rad`须valid_helicity。B减π比较须显式反平行声明及原极性/有效门通过。


In [ ]:
result.table(metrics=['topology', 'helicity', 'relative'])


## 5. 轨迹和速度

$$\mathbf r_u=\mathbf r_w+\mathbf n\odot\mathbf L,\qquad \mathbf v(t_1)=\sum_{k=0}^2\mathbf r(t_k)\ell'_k(t_1).$$

速度使用实际时间戳的三点二次导数，等间隔/非等间隔都支持；同run/segment三连续有效帧才可用，端点NaN。解析真速度为(10,−3)m/s，B−A位移(6,−3)nm，校正相位差0.15rad。静态不是时间序列，不能计算物理速度。缺时间的动态标量按输入索引散点显示，绝不填时间。

center默认是等比例Δx–Δy位移轨迹；时间曲线用`result.plot("center", trajectory=False)`。`trajectory=True`保留绝对XY位置图；没有有效跟踪时位移不可用。拓扑中心单独用topology_center。原始helicity图用π刻度，+π/−π是同向；显式修正图用`result.plot("helicity", corrected=True)`，不会替换原始数据。


In [ ]:
result.plot('velocity', sublattice='both')


## 6. 显式导出

原生成器的analysis.csv保持不变。新CSV导出完整SI审计列，并标source_index；显示筛选不会污染底稿。CSV/SVG重名或与输入冲突一律拒绝，不自动换名。


In [ ]:
result.to_csv(example_dir / 'new-analysis.csv')


In [ ]:
result.plot('center', output=example_dir / 'center.svg')


## 7. 替换为实际数据（按需独立单元）

将明确A/B帧列表代入`pp.analyze`，或仅对可唯一验证绑定最后segment的state调用`pp.analyze(state_path)`。含早期不同stage、同stage不同segment、证据不唯一或冲突时，首版拒绝整个state请求，不靠筛帧/override解读历史。旧OVF的time/PBC/单位/Ms缺项须来源证据，不能猜；单位方向不需要物理Ms，A/m输入需要自己的Ms。

请求级坏容器、全局选项/校准或选择歧义报错，不产生伪成功结果；已提交帧沿用逐帧无效行。查看reason_codes和每指标valid，raw不作有效结果；训练误差包络不是通用物理误差保证。[详细来源合同](../docs/interface-v01/SKYRMION_POSTPROCESSING.md)。
